In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42

df_normalized = pd.read_csv("MERGED_normalized.csv")
df_coda = pd.read_csv("MERGED_coda.csv")

for name, d in [('normalized', df_normalized), ('coda', df_coda)]:
    print(name, d.shape)
    print("  hyg NaN cols:", d.filter(like='hyg').isna().sum().sum())

print(df_normalized['Incidence_next'].describe())
print((df_normalized['Incidence_next'] == 0).sum(), "rows with zero incidence_next")

normalized (985, 132)
  hyg NaN cols: 12199
coda (985, 132)
  hyg NaN cols: 12270
count     985.000000
mean       43.873519
std       217.507774
min         0.000000
25%         0.043693
50%         1.608255
75%        13.479831
max      3432.317650
Name: Incidence_next, dtype: float64
104 rows with zero incidence_next


In [2]:
def add_log_target(source_df):
    source_df = source_df.copy()
    source_df['Incidence_next_log'] = np.log1p(source_df['Incidence_next'])
    return source_df

def build_features(source_df):
    model_df = source_df.copy()
    model_df = pd.get_dummies(model_df, columns=['WHO_Region'], dummy_na=True)
    exclude_cols = ['Country', 'Year', 'Autochthonous_Cases', 'Population',
                     'Incidence', 'Incidence_next', 'Incidence_next_log']
    feature_cols = [c for c in model_df.columns if c not in exclude_cols]
    X = model_df[feature_cols]
    y_log = model_df['Incidence_next_log']
    y_orig = model_df['Incidence_next']
    groups = source_df['Country']
    years = source_df['Year']
    return X, y_log, y_orig, groups, years

def to_original_scale(pred_log):
    return np.clip(np.expm1(pred_log), a_min=0, a_max=None)

df_normalized = add_log_target(df_normalized)
df_coda = add_log_target(df_coda)

feature_sets = {
    'Normalized': build_features(df_normalized),
    'CoDA': build_features(df_coda),
}

model_names = ['XGBoost', 'LightGBM', 'CatBoost']

In [3]:
train_mask = df_normalized['Year'] <= 2019
test_mask = df_normalized['Year'] > 2019

X, y_log, y_orig, groups, years = build_features(df_normalized)

def naive_baselines(y_train_orig, test_incidence_current, y_test_orig):
    results = {}

    valid_mask = test_incidence_current.notna()
    n_dropped = (~valid_mask).sum()

    persistence_pred = test_incidence_current[valid_mask]
    y_test_valid = y_test_orig[valid_mask]
    rmse = np.sqrt(mean_squared_error(y_test_valid, persistence_pred))
    mae = mean_absolute_error(y_test_valid, persistence_pred)
    results['Persistence'] = {'RMSE': rmse, 'MAE': mae, 'n': int(valid_mask.sum())}

    mean_pred = np.full(len(y_test_orig), fill_value=y_train_orig.mean(), dtype=float)
    rmse = np.sqrt(mean_squared_error(y_test_orig, mean_pred))
    mae = mean_absolute_error(y_test_orig, mean_pred)
    results['Mean-predict'] = {'RMSE': rmse, 'MAE': mae, 'n': len(y_test_orig)}

    return results

baseline_results = naive_baselines(
    y_orig[train_mask],
    df_normalized.loc[test_mask, 'Incidence'],
    y_orig[test_mask]
)

for name, scores in baseline_results.items():
    print(f"{name:15s} RMSE={scores['RMSE']:.3f}  MAE={scores['MAE']:.3f}  (n={scores['n']})")

Persistence     RMSE=99.270  MAE=31.496  (n=110)
Mean-predict    RMSE=137.398  MAE=63.987  (n=159)


In [32]:
dropped_mask = test_mask & df_normalized['Incidence'].isna()
kept_mask = test_mask & df_normalized['Incidence'].notna()

print("Incidence_next stats — rows where current Incidence is MISSING:")
print(df_normalized.loc[dropped_mask, 'Incidence_next'].describe())
print("\nIncidence_next stats — rows where current Incidence is PRESENT:")
print(df_normalized.loc[kept_mask, 'Incidence_next'].describe())

Incidence_next stats — rows where current Incidence is MISSING:
count      49.000000
mean       58.041394
std       208.936203
min         0.000000
25%         0.000000
50%         0.085040
75%         3.847139
max      1232.478064
Name: Incidence_next, dtype: float64

Incidence_next stats — rows where current Incidence is PRESENT:
count    110.000000
mean      30.719478
std       89.639544
min        0.000000
25%        0.020232
50%        1.500201
75%       17.552542
max      693.659518
Name: Incidence_next, dtype: float64


In [16]:
import time
import json
import joblib
import os
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

SEED = 42
N_ITER = 40
N_SPLITS = 5

RESULTS_PATH = "tuned_results.json"
MODELS_DIR = "tuned_models"
os.makedirs(MODELS_DIR, exist_ok=True)

def neg_rmse_original(y_true_log, y_pred_log):
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.clip(np.expm1(y_pred_log), a_min=0, a_max=None)
    return -np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))

rmse_scorer = make_scorer(neg_rmse_original, greater_is_better=True)

def get_estimator(name):
    if name == 'XGBoost':
        return XGBRegressor(random_state=SEED, n_jobs=1)
    if name == 'LightGBM':
        return LGBMRegressor(random_state=SEED, n_jobs=1, verbose=-1)
    if name == 'CatBoost':
        return CatBoostRegressor(random_state=SEED, verbose=0, bootstrap_type='Bernoulli')

param_distributions = {
    'XGBoost': {
        'n_estimators': [100, 200, 300, 500, 800],
        'max_depth': [3, 4, 5, 6, 8],
        'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_weight': [1, 3, 5, 7],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [0.1, 1, 5, 10],
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300, 500, 800],
        'max_depth': [3, 4, 5, 6, -1],
        'num_leaves': [15, 31, 63, 127],
        'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
        'min_child_samples': [5, 10, 20, 30],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [0.1, 1, 5, 10],
    },
    'CatBoost': {
        'iterations': [100, 200, 300, 500, 800],
        'depth': [3, 4, 5, 6, 8],
        'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
        'l2_leaf_reg': [1, 3, 5, 7, 9],
        'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
        'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
    }
}

datasets = {'Normalized': df_normalized, 'CoDA': df_coda}
model_names = ['XGBoost', 'LightGBM', 'CatBoost']
total_combos = len(datasets) * len(model_names)

if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, 'r') as f:
        tuned_results = json.load(f)
    print(f"Loaded {len(tuned_results)} previously completed combos from {RESULTS_PATH}")
else:
    tuned_results = {}

tuned_estimators = {}
combo_durations = []
combo_counter = 0
overall_start = time.time()

def make_json_safe(d):
    """Convert numpy scalar types to plain Python types so json.dump doesn't choke."""
    return {k: (v.item() if isinstance(v, np.generic) else v) for k, v in d.items()}

for ds_name, source_df in datasets.items():
    X, y_log, y_orig, groups, years = build_features(source_df)
    gkf = GroupKFold(n_splits=N_SPLITS)
    cv_splits = list(gkf.split(X, y_log, groups=groups))

    for model_name in model_names:
        combo_counter += 1
        key = f"{model_name} ({ds_name})"

        if key in tuned_results:
            print(f"[{combo_counter}/{total_combos}] Skipping {key} — already completed.")
            continue

        combo_start = time.time()
        print(f"\n{'='*70}\n[{combo_counter}/{total_combos}] Tuning {key}...\n{'='*70}")

        estimator = get_estimator(model_name)
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_distributions[model_name],
            n_iter=N_ITER,
            scoring=rmse_scorer,
            cv=cv_splits,
            random_state=SEED,
            n_jobs=-1,
            refit=True,
            verbose=2
        )
        search.fit(X, y_log)

        combo_elapsed = time.time() - combo_start
        combo_durations.append(combo_elapsed)

        tuned_results[key] = {
            'cv_rmse': float(-search.best_score_),
            'best_params': make_json_safe(search.best_params_),
            'dataset': ds_name,
            'model': model_name,
            'n_iter': N_ITER,
            'n_splits': N_SPLITS,
            'seed': SEED
        }
        tuned_estimators[key] = search.best_estimator_

        # --- Save after EVERY combo, not just at the end ---
        with open(RESULTS_PATH, 'w') as f:
            json.dump(tuned_results, f, indent=2)

        model_path = os.path.join(MODELS_DIR, f"{key.replace(' ', '_').replace('(', '').replace(')', '')}.joblib")
        joblib.dump(search.best_estimator_, model_path)

        avg_so_far = np.mean(combo_durations)
        remaining = total_combos - combo_counter - len([k for k in tuned_results if k != key])
        eta_seconds = avg_so_far * max(remaining, 0)

        print(f"\n--> {key} done in {combo_elapsed/60:.1f} min. RMSE={-search.best_score_:.3f}")
        print(f"--> Saved to {RESULTS_PATH} and {model_path}")
        print(f"--> Est. remaining: {eta_seconds/60:.1f} min")

print(f"\nAll combos processed in {(time.time()-overall_start)/60:.1f} min total.")


[1/6] Tuning XGBoost (Normalized)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

--> XGBoost (Normalized) done in 10.9 min. RMSE=179.651
--> Saved to tuned_results.json and tuned_models\XGBoost_Normalized.joblib
--> Est. remaining: 54.6 min

[2/6] Tuning LightGBM (Normalized)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

--> LightGBM (Normalized) done in 4.6 min. RMSE=179.491
--> Saved to tuned_results.json and tuned_models\LightGBM_Normalized.joblib
--> Est. remaining: 23.2 min

[3/6] Tuning CatBoost (Normalized)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

--> CatBoost (Normalized) done in 43.5 min. RMSE=179.873
--> Saved to tuned_results.json and tuned_models\CatBoost_Normalized.joblib
--> Est. remaining: 19.7 min

[4/6] Tuning XGBoost (CoDA)...
Fitting 5 folds for each of 40 candidates, totalling 200 fits

--> XGBoost (CoDA) done in 10.5 min. RMSE=179.610
--> Saved to tuned_results.json and tuned_models\XGBoost_CoDA.jobli

In [33]:
from sklearn.model_selection import cross_val_predict

oof_results = {}

for key in tuned_results:
    ds_name = tuned_results[key]['dataset']
    model_name = tuned_results[key]['model']
    source_df = df_normalized if ds_name == 'Normalized' else df_coda
    X, y_log, y_orig, groups, years = build_features(source_df)

    gkf = GroupKFold(n_splits=N_SPLITS)
    cv_splits = list(gkf.split(X, y_log, groups=groups))

    estimator = get_estimator(model_name)
    estimator.set_params(**tuned_results[key]['best_params'])

    oof_pred_log = cross_val_predict(estimator, X, y_log, cv=cv_splits, n_jobs=-1)
    oof_pred_orig = np.clip(np.expm1(oof_pred_log), a_min=0, a_max=None)

    mae = np.mean(np.abs(y_orig.values - oof_pred_orig))
    rmse = np.sqrt(np.mean((y_orig.values - oof_pred_orig) ** 2))

    oof_results[key] = {'MAE': mae, 'RMSE': rmse}
    print(f"{key:28s}  OOF MAE={mae:.3f}  OOF RMSE={rmse:.3f}")

XGBoost (Normalized)          OOF MAE=43.597  OOF RMSE=220.408
LightGBM (Normalized)         OOF MAE=43.421  OOF RMSE=220.317
CatBoost (Normalized)         OOF MAE=43.168  OOF RMSE=220.465
XGBoost (CoDA)                OOF MAE=43.906  OOF RMSE=220.487
LightGBM (CoDA)               OOF MAE=43.855  OOF RMSE=220.488
CatBoost (CoDA)               OOF MAE=43.903  OOF RMSE=220.430


In [34]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_predict

X, y_log, y_orig, groups, years = build_features(df_normalized)
gkf = GroupKFold(n_splits=N_SPLITS)
cv_splits = list(gkf.split(X, y_log, groups=groups))

dummy_oof_pred_log = cross_val_predict(DummyRegressor(strategy='mean'), X, y_log, cv=cv_splits)
dummy_oof_pred_orig = np.clip(np.expm1(dummy_oof_pred_log), a_min=0, a_max=None)

dummy_mae = np.mean(np.abs(y_orig.values - dummy_oof_pred_orig))
dummy_rmse = np.sqrt(np.mean((y_orig.values - dummy_oof_pred_orig) ** 2))
print(f"Dummy (predict training-fold mean)   OOF MAE={dummy_mae:.3f}  OOF RMSE={dummy_rmse:.3f}")

Dummy (predict training-fold mean)   OOF MAE=43.882  OOF RMSE=221.088


In [38]:
temporal_results = {}
temporal_results_110 = {}

for ds_name, (X, y_log, y_orig, groups, years) in feature_sets.items():
    train_mask_ds = years <= 2019
    test_mask_ds = years > 2019

    for model_name in model_names:
        key = f"{model_name} ({ds_name})"
        estimator = get_estimator(model_name)
        estimator.set_params(**tuned_results[key]['best_params'])
        estimator.fit(X[train_mask_ds], y_log[train_mask_ds])

        pred_log = estimator.predict(X[test_mask_ds])
        pred_orig = to_original_scale(pred_log)

        rmse = np.sqrt(mean_squared_error(y_orig[test_mask_ds], pred_orig))
        mae = mean_absolute_error(y_orig[test_mask_ds], pred_orig)
        temporal_results[key] = {'RMSE': rmse, 'MAE': mae, 'n_test': int(test_mask_ds.sum())}

        pred_orig_110 = pred_orig[valid_mask_110.values]
        y_orig_110 = y_orig[test_mask_ds].values[valid_mask_110.values]
        rmse_110 = np.sqrt(mean_squared_error(y_orig_110, pred_orig_110))
        mae_110 = mean_absolute_error(y_orig_110, pred_orig_110)
        temporal_results_110[key] = {'RMSE': rmse_110, 'MAE': mae_110, 'n_test': int(valid_mask_110.sum())}

        print(f"{key:28s}  Full (n={test_mask_ds.sum()}): RMSE={rmse:6.3f} MAE={mae:6.3f}   |   "
              f"vs. Persistence subset (n={valid_mask_110.sum()}): RMSE={rmse_110:6.3f} MAE={mae_110:6.3f}")

with open("temporal_results.json", 'w') as f:
    json.dump(temporal_results, f, indent=2)

XGBoost (Normalized)          Full (n=159): RMSE=139.070 MAE=37.374   |   vs. Persistence subset (n=110): RMSE=91.657 MAE=28.203
LightGBM (Normalized)         Full (n=159): RMSE=138.463 MAE=36.858   |   vs. Persistence subset (n=110): RMSE=91.476 MAE=27.735
CatBoost (Normalized)         Full (n=159): RMSE=137.674 MAE=36.449   |   vs. Persistence subset (n=110): RMSE=91.012 MAE=27.332
XGBoost (CoDA)                Full (n=159): RMSE=139.337 MAE=36.337   |   vs. Persistence subset (n=110): RMSE=90.901 MAE=26.907
LightGBM (CoDA)               Full (n=159): RMSE=140.544 MAE=37.148   |   vs. Persistence subset (n=110): RMSE=90.802 MAE=27.051
CatBoost (CoDA)               Full (n=159): RMSE=137.663 MAE=36.476   |   vs. Persistence subset (n=110): RMSE=91.105 MAE=27.514


In [39]:
rows = []
for key in tuned_results:
    rows.append({
        'Model': tuned_results[key]['model'],
        'Feature Set': tuned_results[key]['dataset'],
        'Grouped-CV OOF RMSE': oof_results[key]['RMSE'],
        'Grouped-CV OOF MAE': oof_results[key]['MAE'],
        'Temporal RMSE (full, n=159)': temporal_results[key]['RMSE'],
        'Temporal MAE (full, n=159)': temporal_results[key]['MAE'],
        'Temporal RMSE (n=110 subset)': temporal_results_110[key]['RMSE'],
        'Temporal MAE (n=110 subset)': temporal_results_110[key]['MAE'],
    })

comparison_df = pd.DataFrame(rows).sort_values('Temporal RMSE (full, n=159)')

reference_rows = pd.DataFrame([
    {'Model': 'Dummy (grouped-CV mean)', 'Feature Set': '-',
     'Grouped-CV OOF RMSE': dummy_rmse, 'Grouped-CV OOF MAE': dummy_mae,
     'Temporal RMSE (full, n=159)': np.nan, 'Temporal MAE (full, n=159)': np.nan,
     'Temporal RMSE (n=110 subset)': np.nan, 'Temporal MAE (n=110 subset)': np.nan},
    {'Model': 'Mean-predict', 'Feature Set': '-',
     'Grouped-CV OOF RMSE': np.nan, 'Grouped-CV OOF MAE': np.nan,
     'Temporal RMSE (full, n=159)': baseline_results['Mean-predict']['RMSE'],
     'Temporal MAE (full, n=159)': baseline_results['Mean-predict']['MAE'],
     'Temporal RMSE (n=110 subset)': np.nan, 'Temporal MAE (n=110 subset)': np.nan},
    {'Model': 'Persistence', 'Feature Set': '-',
     'Grouped-CV OOF RMSE': np.nan, 'Grouped-CV OOF MAE': np.nan,
     'Temporal RMSE (full, n=159)': np.nan, 'Temporal MAE (full, n=159)': np.nan,
     'Temporal RMSE (n=110 subset)': baseline_results['Persistence']['RMSE'],
     'Temporal MAE (n=110 subset)': baseline_results['Persistence']['MAE']},
])

comparison_df = pd.concat([comparison_df, reference_rows], ignore_index=True)
comparison_df.to_csv("model_comparison_final.csv", index=False)
comparison_df

,Model,Feature Set,Grouped-CV OOF RMSE,Grouped-CV OOF MAE,"Temporal RMSE (full, n=159)","Temporal MAE (full, n=159)",Temporal RMSE (n=110 subset),Temporal MAE (n=110 subset)
0,CatBoost,CoDA,220.429906,43.902867,137.662966,36.476375,91.105198,27.514370
1,CatBoost,Normalized,220.465339,43.168091,137.674133,36.449212,91.011977,27.332221
2,LightGBM,Normalized,220.317272,43.421424,138.462545,36.858432,91.476467,27.734516
3,XGBoost,Normalized,220.407653,43.597047,139.070211,37.374093,91.656785,28.202937
4,XGBoost,CoDA,220.486795,43.906038,139.337399,36.337262,90.900789,26.906594
5,LightGBM,CoDA,220.487714,43.854841,140.543516,37.148419,90.801789,27.050924
6,Dummy (grouped-CV mean),-,221.087565,43.881522,NaN,NaN,NaN,NaN
7,Mean-predict,-,NaN,NaN,137.397714,63.986906,NaN,NaN
8,Persistence,-,NaN,NaN,NaN,NaN,99.270087,31.495660
